<a href="https://colab.research.google.com/github/Realbenjizo/python-functions-for-data-engineers/blob/main/Chapter_5_Decorators_for_Cross_Cutting_Concerns.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import logging

In [ ]:
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

In [ ]:
# Replace link with your link to your downloaded file
transactions = "/content/drive/MyDrive/data_files/transactions.csv"

transactions_df = pd.read_csv(transactions)
print(transactions_df)

   user_id  amount transaction_date     status
0      1.0  250.50        1 03 2025  completed
1      2.0    0.00        2 03 2025    pending
2      1.0  175.75        1 03 2025  completed
3      3.0   89.99        3 03 2025  completed
4      2.0  -10.00        2 03 2025     failed
5      4.0  320.00        4 03 2025  completed
6      NaN  150.00        1 03 2025  completed
7      1.0  200.00        5 03 2025  completed


In [ ]:
from typing import Dict, Optional

user_lookup: Dict[int, str] = {
    1: "Alice Johnson",
    2: "Bob Smith",
    3: "Carol Williams",
    4: "David Brown"
}

In [ ]:
import time
import logging
from typing import Dict, Optional
import pandas as pd

logger = logging.getLogger(__name__)

def enrich_user_names(df: pd.DataFrame,
                      user_lookup: Optional[Dict[int, str]] = None
                      ) -> pd.DataFrame:
    """Enriching DataFrame with user names"""

    logger.info("Starting enrich_user_names")  # Manual Infrastructure Log
    start = time.perf_counter()                # Manual Infrastructure Timing

    try:
        df = df.copy()
        if user_lookup is None:
            df["user_name"] = "Unknown"
            return df

        # The core business function
        df["user_name"] = df["user_id"].map(user_lookup).fillna("Unknown")

        duration = time.perf_counter() - start
        logger.info(f"Finished enrich_user_names in: {duration:.2f} seconds")
        return df
    except Exception as e:
        raise RuntimeError("Error while enriching user names.") from e

enrich_user_names(transactions_df, user_lookup)

,user_id,amount,transaction_date,status,user_name
0,1.0,250.50,1 03 2025,completed,Alice Johnson
1,2.0,0.00,2 03 2025,pending,Bob Smith
2,1.0,175.75,1 03 2025,completed,Alice Johnson
3,3.0,89.99,3 03 2025,completed,Carol Williams
4,2.0,-10.00,2 03 2025,failed,Bob Smith
5,4.0,320.00,4 03 2025,completed,David Brown
6,NaN,150.00,1 03 2025,completed,Unknown
7,1.0,200.00,5 03 2025,completed,Alice Johnson


## **Decorator**

In [ ]:
import functools
import logging
from typing import Callable, Any

logger = logging.getLogger(__name__)

def log_step(func: Callable[..., Any]) -> Callable[..., Any]:
    """Decorator that logs when a pipeline step starts and finishes successfully."""
    @functools.wraps(func)
    def wrapper(*args: Any, **kwargs: Any) -> Any:
        logger.info(f"Starting step: '{func.__name__}'")

        # Execute the wrapped core data transformation step
        result = func(*args, **kwargs)

        # This only executes if the inner function completes without an exception
        logger.info(f"Finished step: '{func.__name__}' successfully.")
        return result
    return wrapper

# **Decorator to pipeline**

In [ ]:
@log_step
def enrich_user_names(
    df: pd.DataFrame,
    user_lookup: Optional[Dict[int, str]] = None
) -> pd.DataFrame:
    """Only the transformation; no logging code here."""
    df = df.copy()
    if user_lookup is None:
        df['user_name'] = 'Unknown'
        return df
    df['user_name'] = df['user_id'].map(user_lookup).fillna('Unknown')
    return df

if __name__ == "__main__":
    print(enrich_user_names(transactions_df, user_lookup))

   user_id  amount transaction_date     status       user_name
0      1.0  250.50        1 03 2025  completed   Alice Johnson
1      2.0    0.00        2 03 2025    pending       Bob Smith
2      1.0  175.75        1 03 2025  completed   Alice Johnson
3      3.0   89.99        3 03 2025  completed  Carol Williams
4      2.0  -10.00        2 03 2025     failed       Bob Smith
5      4.0  320.00        4 03 2025  completed     David Brown
6      NaN  150.00        1 03 2025  completed         Unknown
7      1.0  200.00        5 03 2025  completed   Alice Johnson


# **Adding More behavior**

In [ ]:
import functools
import logging
import time
from typing import Callable, Any

logger = logging.getLogger(__name__)

def pipeline_step(func: Callable[..., Any]) -> Callable[..., Any]:
    """Decorator that adds logging, execution timing, and error reporting."""

    @functools.wraps(func)
    def wrapper(*args: Any, **kwargs: Any) -> Any:
        func_name = func.__name__
        logger.info(f"Starting: {func_name}")

        start_time = time.perf_counter()
        try:
            result = func(*args, **kwargs)
        except Exception:
            duration = time.perf_counter() - start_time
            logger.exception(
                f"Pipeline step failed: {func_name} "
                f"after {duration:.2f} seconds."
            )
            raise
        else:
            duration = time.perf_counter() - start_time
            logger.info(f"Finished: {func_name} in {duration:.2f} seconds")
            return result
    return wrapper

# **Applying decorator**

In [ ]:
@pipeline_step
def enrich_user_names(
    df: pd.DataFrame,
    user_lookup: Optional[Dict[int, str]] = None
) -> pd.DataFrame:
    """Mapping user_ids to names."""
    df = df.copy()
    if user_lookup is None:
        df['user_name'] = 'Unknown'
        return df
    df['user_name'] = df['user_id'].map(user_lookup).fillna('Unknown')
    return df

if __name__ == "__main__":
    print(enrich_user_names(transactions_df, user_lookup=user_lookup))

   user_id  amount transaction_date     status       user_name
0      1.0  250.50        1 03 2025  completed   Alice Johnson
1      2.0    0.00        2 03 2025    pending       Bob Smith
2      1.0  175.75        1 03 2025  completed   Alice Johnson
3      3.0   89.99        3 03 2025  completed  Carol Williams
4      2.0  -10.00        2 03 2025     failed       Bob Smith
5      4.0  320.00        4 03 2025  completed     David Brown
6      NaN  150.00        1 03 2025  completed         Unknown
7      1.0  200.00        5 03 2025  completed   Alice Johnson


# **Parameterized decorator**

In [ ]:
def with_retry(max_attempts: int = 3) -> Callable[..., Any]:
    """Decorator that retries a function if it fails."""
    def decorator(func: Callable[..., Any]) -> Callable[..., Any]:
        @functools.wraps(func)
        def wrapper(*args: Any, **kwargs: Any):
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    logger.warning(f"Attempt {attempt} failed for {func.__name__}")
                    if attempt == max_attempts:
                        raise RuntimeError(
                            f"{func.__name__} failed after"
                            f"{max_attempts} attempts."
                        )
                    time.sleep(2 ** attempt) # A simple backoff
        return wrapper

    return decorator

In [ ]:
import requests

@with_retry(max_attempts=3)
def fetch_user_from_api(user_id: int) -> Optional[Dict]:
    """Fetch a single user from API."""
    url = f"https://jsonplaceholder.typicode.com/users/{user_id}"
    response = requests.get(url, timeout=5)
    response.raise_for_status()

    return response.json()

if __name__ == "__main__":
    fetch_user_from_api(5)

# **Stacking the decorators**

In [ ]:
import requests

@with_retry(max_attempts=3)
@pipeline_step
def fetch_user_from_api(user_id: int) -> Optional[Dict]:
    """Fetch a single user from API."""
    url = f"https://jsonplaceholder.typicode.com/users/{user_id}"
    response = requests.get(url, timeout=5)
    response.raise_for_status()
    return response.json()

if __name__ == "__main__":
    result = fetch_user_from_api(5)

In [ ]:
fetch_user_from_api = with_retry(max_attempts=5)(
    pipeline_step(fetch_user_from_api)
)

# **Wrapper**

In [ ]:
def fetch_user_from_api_with_retry(user_id: int) -> Optional[Dict]:
    """Fetch user from API with retry logic applied at call time."""
    # Create the retry decorator with configuration
    retry_decorator = with_retry(max_attempts=3)

    # Apply the decorator to the function
    wrapped_fetch = retry_decorator(fetch_user_from_api)

    # Call the wrapped function
    return wrapped_fetch(user_id)

# retry is applied only when you call this wrapper
if __name__ == "__main__":
    result = fetch_user_from_api_with_retry(5)

## **Challenge**

In [ ]:
import logging

logger = logging.getLogger(__name__)

def process_transactions(df: pd.DataFrame,
                         lookup: dict) -> pd.DataFrame:
    """Transactions processing by dropping nulls and enriching with user names."""

    logger.info(f"Starting process_transactions with {len(df)} rows")
    start_time = time.perf_time()
    try:
        # Drop null values
        df_cleaned = df.dropna()
        # Enrich with user names
        df_cleaned['user_name'] = df_cleaned['user_id'].map(lookup)
        elapsed = time.perf_time() - start_time
        logger.info(f"Finished process_transactions in {elapsed:.2f} seconds")
        return df_cleaned

    except Exception as e:
       raise RuntimeError(f"Error in process_transactions: {type(e).__name__}: {e}") from e

In [ ]:
@log_step
def process_transactions(df: pd.DataFrame, lookup: dict) -> pd.DataFrame:
    """Transactions processing by dropping nulls and enriching with user names."""
    df_cleaned = df.dropna()
    df_cleaned['user_name'] = df_cleaned['user_id'].map(lookup)
    return df_cleaned

## **Solution**

In [ ]:
import logging
import time
import functools
from typing import Callable, Any

logger = logging.getLogger(__name__)

def log_step(func: Callable[..., Any]) -> Callable[..., Any]:
    """Decorator that handles logging, timing, and error wrapping for a pipeline step."""
    @functools.wraps(func)
    def wrapper(*args: Any, **kwargs: Any) -> Any:
        func_name = func.__name__
        logger.info(f"Starting {func_name}...")

        start_time = time.perf_counter()
        try:
            # Execute the core transformation logic
            result = func(*args, **kwargs)

            # Log success with timing
            elapsed = time.perf_counter() - start_time
            logger.info(f"Finished {func_name} in {elapsed:.2f} seconds")
            return result
        except Exception as e:
            # Wrap the error and preserve the traceback
            raise RuntimeError(f"Error in {func_name}: {type(e).__name__}: {e}") from e
    return wrapper


@log_step
def process_transactions(df: pd.DataFrame, lookup: dict) -> pd.DataFrame:
    """Transactions processing by dropping nulls and enriching with user names."""
    df_cleaned = df.dropna().copy()
    df_cleaned['user_name'] = df_cleaned['user_id'].map(lookup)
    return df_cleaned


if __name__ == "__main__":
    processed_data = process_transactions(transactions_df, lookup=user_lookup)
    print(processed_data)

   user_id  amount transaction_date     status       user_name
0      1.0  250.50        1 03 2025  completed   Alice Johnson
1      2.0    0.00        2 03 2025    pending       Bob Smith
2      1.0  175.75        1 03 2025  completed   Alice Johnson
3      3.0   89.99        3 03 2025  completed  Carol Williams
4      2.0  -10.00        2 03 2025     failed       Bob Smith
5      4.0  320.00        4 03 2025  completed     David Brown
7      1.0  200.00        5 03 2025  completed   Alice Johnson
